# 01 — Baseline PPO on LunarLander-v3

Train a standard PPO agent using the environment's built-in reward, evaluate its performance, and log results to SQLite.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from pathlib import Path

from lunarlander.db_logger import ExperimentLogger

# Paths
CHECKPOINT_DIR = Path('../checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
DB_PATH = Path('../experiments.db')

print('Setup complete.')

In [ ]:
# Hyperparameters
TOTAL_TIMESTEPS = 500_000
N_STEPS = 2048
BATCH_SIZE = 64
N_EPOCHS = 10
LEARNING_RATE = 3e-4
GAMMA = 0.999
GAE_LAMBDA = 0.98
ENT_COEF = 0.01
SEED = 42
N_EVAL_EPISODES = 50

hyperparams = {
    'total_timesteps': TOTAL_TIMESTEPS,
    'n_steps': N_STEPS,
    'batch_size': BATCH_SIZE,
    'n_epochs': N_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'gamma': GAMMA,
    'gae_lambda': GAE_LAMBDA,
    'ent_coef': ENT_COEF,
    'seed': SEED,
}
print('Hyperparameters:', hyperparams)

In [ ]:
# Create environment
def make_env():
    env = gym.make('LunarLander-v3')
    env = Monitor(env)
    return env

env = DummyVecEnv([make_env])
print('Env created. Obs shape:', env.observation_space.shape, 'Action space:', env.action_space)

In [ ]:
# Train PPO
model = PPO(
    'MlpPolicy',
    env,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    learning_rate=LEARNING_RATE,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    ent_coef=ENT_COEF,
    verbose=1,
    seed=SEED,
)

print(f'Training for {TOTAL_TIMESTEPS:,} timesteps...')
model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
print('Training complete!')

In [ ]:
# Evaluate
eval_env = DummyVecEnv([make_env])

mean_reward, std_reward = evaluate_policy(
    model, eval_env, n_eval_episodes=N_EVAL_EPISODES, deterministic=True
)
print(f'Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}')

In [ ]:
# Compute success rate (reward > 200) and crash rate
raw_env = gym.make('LunarLander-v3')
episode_rewards = []
landed_count = 0
crashed_count = 0
ep_lengths = []

for _ in range(N_EVAL_EPISODES):
    obs, _ = raw_env.reset()
    ep_reward = 0.0
    ep_len = 0
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = raw_env.step(action)
        ep_reward += reward
        ep_len += 1
        done = terminated or truncated
        if terminated:
            # Positive landing reward means successful landing
            if reward == 100:
                landed_count += 1
            elif reward == -100:
                crashed_count += 1
    episode_rewards.append(ep_reward)
    ep_lengths.append(ep_len)

success_rate = landed_count / N_EVAL_EPISODES
crash_rate = crashed_count / N_EVAL_EPISODES
mean_ep_len = float(np.mean(ep_lengths))

print(f'Success rate (landed): {success_rate:.2%}')
print(f'Crash rate:            {crash_rate:.2%}')
print(f'Mean episode length:   {mean_ep_len:.1f} steps')
raw_env.close()

In [ ]:
# Save checkpoint
checkpoint_path = CHECKPOINT_DIR / 'baseline_ppo'
model.save(str(checkpoint_path))
print(f'Checkpoint saved to {checkpoint_path}.zip')

In [ ]:
# Log to SQLite
logger = ExperimentLogger(DB_PATH)
row_id = logger.log(
    name='Baseline PPO — LunarLander-v3',
    exp_type='baseline_ppo',
    mean_reward=float(mean_reward),
    std_reward=float(std_reward),
    success_rate=success_rate,
    crash_rate=crash_rate,
    mean_ep_len=mean_ep_len,
    hyperparams=hyperparams,
    notes=f'Standard SB3 PPO on LunarLander-v3, {TOTAL_TIMESTEPS} timesteps.',
)
print(f'Logged to SQLite, row id={row_id}')

# Verify
rows = logger.fetch_all()
print(f'Total experiments in DB: {len(rows)}')
for r in rows:
    print(f"  [{r['id']}] {r['name']} | mean_reward={r['mean_reward']}")

In [ ]:
# Quick reward distribution plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(episode_rewards, bins=20, edgecolor='black', color='steelblue', alpha=0.8)
ax.axvline(mean_reward, color='red', linestyle='--', label=f'Mean={mean_reward:.1f}')
ax.axvline(200, color='green', linestyle=':', label='Success threshold (200)')
ax.set_xlabel('Episode Return')
ax.set_ylabel('Count')
ax.set_title('Baseline PPO — Episode Return Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/baseline_ppo_rewards.png', dpi=100)
plt.show()
print('Plot saved.')